# Notebook 10: Model Calibration and Thresholds

## Why This Matters

A model that outputs probabilities is only useful if those probabilities mean something. Calibration asks: when your model says "70% probability of fraud", does fraud actually occur 70% of the time? Uncalibrated models make systematic decisions worse -- especially in cost-sensitive settings (fraud, medical diagnosis, ad bidding) where the probability is used directly as an input to a decision policy. Beyond calibration, the default 0.5 decision threshold is almost never optimal: you should select thresholds using the actual cost structure of your problem.

## Background

**Reliability diagram**: Plot observed frequency vs. predicted probability in bins. A perfectly calibrated model lies on the diagonal.

**ECE (Expected Calibration Error)**: Weighted average of |accuracy - confidence| across bins. Values < 0.02 are generally good.

**Platt scaling**: Fit a logistic regression on top of model scores using a held-out calibration set.

**Temperature scaling**: Single parameter T divides logits: p = softmax(logit/T). Preserves ranking, only adjusts confidence.

**Isotonic regression**: Non-parametric monotone calibration. More flexible than Platt but requires more calibration data (>1000 samples).

## Community Context

NeurIPS 2017 "On Calibration of Modern Neural Networks" (Guo et al.) showed that modern deep networks are severely overconfident. Random forests are typically well-calibrated; gradient boosting and SVMs tend to be overconfident. sklearn's `CalibratedClassifierCV` handles both Platt and isotonic calibration with proper cross-validation.

In [ ]:
%pip install -q scikit-learn xgboost numpy pandas matplotlib scipy

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.metrics import brier_score_loss, roc_auc_score
import xgboost as xgb

np.random.seed(42)

X, y = make_classification(
    n_samples=5000, n_features=20, n_informative=10,
    n_redundant=5, n_clusters_per_class=2, flip_y=0.05, random_state=42
)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, stratify=y, random_state=42
)
X_cal, X_eval, y_cal, y_eval = train_test_split(
    X_test, y_test, test_size=0.5, stratify=y_test, random_state=42
)
print(f"Train: {X_train.shape}, Cal: {X_cal.shape}, Eval: {X_eval.shape}")

## 1. Calibration Curves and ECE

The reliability diagram (calibration curve) reveals systematic over/under-confidence. A model above the diagonal is underconfident; below is overconfident. Tree ensembles, especially gradient boosting, are systematically overconfident because they push predictions toward the extremes.

In [ ]:
def compute_ece(y_true, y_prob, n_bins=10):
    bins = np.linspace(0, 1, n_bins + 1)
    ece = 0.0
    for i in range(n_bins):
        mask = (y_prob >= bins[i]) & (y_prob < bins[i+1])
        if mask.sum() == 0:
            continue
        bin_acc = y_true[mask].mean()
        bin_conf = y_prob[mask].mean()
        ece += mask.mean() * abs(bin_acc - bin_conf)
    return ece

models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42),
    "GradientBoosting": GradientBoostingClassifier(n_estimators=100, random_state=42),
    "XGBoost": xgb.XGBClassifier(n_estimators=100, eval_metric="logloss", verbosity=0, random_state=42)
}

results = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    probs = model.predict_proba(X_eval)[:, 1]
    ece = compute_ece(y_eval, probs)
    brier = brier_score_loss(y_eval, probs)
    auc = roc_auc_score(y_eval, probs)
    results[name] = {"model": model, "probs": probs, "ECE": ece, "Brier": brier, "AUC": auc}
    print(f"{name:25s}  AUC={auc:.4f}  ECE={ece:.4f}  Brier={brier:.4f}")

fig, ax = plt.subplots(figsize=(8, 6))
ax.plot([0, 1], [0, 1], "k--", label="Perfect calibration")
for name, r in results.items():
    ece_val = r["ECE"]
    frac_pos, mean_pred = calibration_curve(y_eval, r["probs"], n_bins=10)
    ax.plot(mean_pred, frac_pos, "o-", label=f"{name} (ECE={ece_val:.3f})")
ax.set_xlabel("Mean predicted probability")
ax.set_ylabel("Fraction of positives")
ax.set_title("Calibration Curves")
ax.legend(loc="upper left", fontsize=8)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 2. Platt Scaling and Isotonic Regression

Platt scaling fits `sigmoid(a * score + b)` on a held-out calibration set. It is fast and works well when miscalibration is roughly sigmoidal. Isotonic regression is more flexible but needs more calibration data.

In [ ]:
from sklearn.base import clone

calibrated_models = {}
for name, r in results.items():
    base_model = clone(r["model"])
    base_model.fit(X_train, y_train)
    cal_platt = CalibratedClassifierCV(base_model, method="sigmoid", cv="prefit")
    cal_platt.fit(X_cal, y_cal)
    probs_platt = cal_platt.predict_proba(X_eval)[:, 1]
    cal_iso = CalibratedClassifierCV(base_model, method="isotonic", cv="prefit")
    cal_iso.fit(X_cal, y_cal)
    probs_iso = cal_iso.predict_proba(X_eval)[:, 1]
    ece_orig = compute_ece(y_eval, r["probs"])
    ece_platt = compute_ece(y_eval, probs_platt)
    ece_iso = compute_ece(y_eval, probs_iso)
    calibrated_models[name] = {"platt": probs_platt, "isotonic": probs_iso}
    print(f"{name:25s}  ECE: {ece_orig:.4f} -> Platt: {ece_platt:.4f} -> Isotonic: {ece_iso:.4f}")

## 3. Temperature Scaling

Temperature scaling divides logits by a scalar T before the sigmoid. T > 1 softens predictions (reduces overconfidence), T < 1 sharpens them. It preserves ranking (AUC unchanged) and has just one parameter to fit, making it popular for neural networks.

In [ ]:
from scipy.optimize import minimize_scalar
from scipy.special import expit
from sklearn.metrics import log_loss

def apply_temperature(logits, T):
    return expit(logits / T)

gb = results["GradientBoosting"]["model"]
raw_scores_cal = gb.decision_function(X_cal)
raw_scores_eval = gb.decision_function(X_eval)

def temperature_loss(T):
    probs = apply_temperature(raw_scores_cal, T)
    return log_loss(y_cal, probs)

result = minimize_scalar(temperature_loss, bounds=(0.1, 10.0), method="bounded")
T_opt = result.x
print(f"Optimal temperature T = {T_opt:.4f}")

probs_uncal = expit(raw_scores_eval)
probs_temp = apply_temperature(raw_scores_eval, T_opt)

ece_uncal = compute_ece(y_eval, probs_uncal)
ece_temp = compute_ece(y_eval, probs_temp)
auc_uncal = roc_auc_score(y_eval, probs_uncal)
auc_temp = roc_auc_score(y_eval, probs_temp)

print(f"Uncalibrated:    ECE={ece_uncal:.4f}, AUC={auc_uncal:.4f}")
print(f"Temp scaled:     ECE={ece_temp:.4f}, AUC={auc_temp:.4f}")
print(f"AUC preserved: {abs(auc_uncal - auc_temp) < 0.001}")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, (probs, label) in zip(axes, [(probs_uncal, "Before"), (probs_temp, f"After (T={T_opt:.2f})")]):
    frac_pos, mean_pred = calibration_curve(y_eval, probs, n_bins=10)
    ax.plot([0, 1], [0, 1], "k--")
    ax.plot(mean_pred, frac_pos, "o-r")
    ax.set_title(f"{label} Temperature Scaling")
    ax.set_xlabel("Predicted probability")
    ax.set_ylabel("Fraction of positives")
    ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 4. Threshold Selection with Cost Matrix

In real applications, false negatives and false positives have different costs. Setting the threshold to 0.5 optimizes nothing in particular. Given `fn_cost` and `fp_cost`, the optimal threshold minimizes expected cost. Fraud detection needs high recall (low fn_cost threshold); alert triage needs high precision (higher threshold to avoid fatigue).

In [ ]:
from sklearn.metrics import confusion_matrix

gb_probs = apply_temperature(raw_scores_eval, T_opt)

scenarios = [
    ("Equal cost (0.5)", 1.0, 1.0),
    ("Fraud detection (FN expensive)", 10.0, 1.0),
    ("Alert fatigue (FP expensive)", 1.0, 5.0),
]

thresholds = np.linspace(0.01, 0.99, 200)
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

print(f"Threshold analysis (n_eval={len(y_eval)}):")
header = f"{"Scenario":40s} {"Threshold":>10s} {"TotalCost":>12s} {"Recall":>8s} {"Precision":>10s}"
print(header)
print("-" * 85)

for ax, (scenario_name, fn_cost, fp_cost) in zip(axes, scenarios):
    costs = []
    for t in thresholds:
        y_pred = (gb_probs >= t).astype(int)
        tn, fp, fn, tp = confusion_matrix(y_eval, y_pred).ravel()
        costs.append(fn * fn_cost + fp * fp_cost)
    costs = np.array(costs)
    opt_idx = np.argmin(costs)
    opt_thresh = thresholds[opt_idx]
    opt_cost = costs[opt_idx]
    y_pred_opt = (gb_probs >= opt_thresh).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_eval, y_pred_opt).ravel()
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    print(f"{scenario_name:40s} {opt_thresh:>10.3f} {opt_cost:>12.1f} {recall:>8.3f} {precision:>10.3f}")
    ax.plot(thresholds, costs, "b-")
    ax.axvline(opt_thresh, color="r", linestyle="--", label=f"Opt={opt_thresh:.2f}")
    ax.set_xlabel("Threshold")
    ax.set_ylabel("Total Cost")
    ax.set_title(f"{scenario_name}\n(FN={fn_cost:.0f}x, FP={fp_cost:.0f}x)")
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Summary

| Method | Parameters | Best For | Limitation |
|--------|-----------|----------|------------|
| Platt Scaling | 2 (a, b) | Overconfident models | Assumes sigmoid miscalibration |
| Temperature Scaling | 1 (T) | Neural networks | Assumes uniform overconfidence |
| Isotonic Regression | Non-parametric | Flexible miscalibration | Needs 1000+ calibration samples |
| Default threshold (0.5) | 0 | Balanced equal-cost problems | Rarely optimal in practice |
| Cost-optimal threshold | 0 | Cost-sensitive applications | Requires business cost estimates |

**Interview tip:** Calibration and AUC are orthogonal -- a model can have high AUC but be badly miscalibrated. AUC measures ranking quality; calibration measures probability accuracy. For systems that use model outputs as inputs to optimization (ad bidding, recommendation scores), calibration is as important as AUC.